# AI Text Detector —
**Architecture**: RoBERTa with two head
- `binary_head`: Human (0) vs AI (1)
- `model_head`: which AI model (only for text)

**How the labels are mapped in the dataset:**
```
0 = Human
1 = ChatGPT
2 = Gemini
3 = Grok
4 = Claude
5 = DeepSeek
```

In [2]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
# =============================================
# 1. IMPORTS
# =============================================
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [4]:
# =============================================
#  LABEL MAPPING
# =============================================
#
#   0=Human, 1=ChatGPT, 2=Gemini, 3=Grok, 4=Claude, 5=DeepSeek

HUMAN_ID   = 0
NUM_MODELS = 5  # models AI to classify

# Remapping for the ID
AI_ID_TO_MODEL_IDX = {1: 0,
                      2: 1,
                      3: 2,
                      4: 3,
                      5: 4}
MODEL_IDX_TO_NAME  = {0: 'ChatGPT',
                      1: 'Gemini',
                      2: 'Grok',
                      3: 'Claude',
                      4: 'DeepSeek'}
ID_TO_NAME         = {0: 'Human',
                      1: 'ChatGPT',
                      2: 'Gemini',
                      3: 'Grok',
                      4: 'Claude',
                      5: 'DeepSeek'}

print('AI_ID_TO_MODEL_IDX:', AI_ID_TO_MODEL_IDX)
print('MODEL_IDX_TO_NAME: ', MODEL_IDX_TO_NAME)

AI_ID_TO_MODEL_IDX: {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
MODEL_IDX_TO_NAME:  {0: 'ChatGPT', 1: 'Gemini', 2: 'Grok', 3: 'Claude', 4: 'DeepSeek'}


In [5]:
df = pd.read_csv('/content/drive/MyDrive/malto/train.csv')

### DATASET INFO

In [8]:
import plotly.express as px

# Map numeric labels to names
df["label_name"] = df["LABEL"].map(ID_TO_NAME)
df["text_length"] = df["TEXT"].astype(str).apply(lambda x: len(x.split()))

stats = df.groupby("label_name")["text_length"].describe()
print(stats)

fig_hist = px.histogram(
    df,
    x="text_length",
    color="label_name",
    nbins=50,
    barmode="overlay",
    opacity=0.6,
    title="Text Length Distribution by Source",
    labels={"text_length": "Text Length (characters)", "label_name": "Source"}
)

fig_box = px.box(
    df,
    x="label_name",
    y="text_length",
    color="label_name",
    title="Text Length Comparison by Source",
    labels={"text_length": "Text Length", "label_name": "Source"}
)

fig_violin = px.violin(
    df,
    x="label_name",
    y="text_length",
    color="label_name",
    box=True,
    points="outliers",
    title="Text Length Distribution Shape by Source"
)

fig_hist.show()
fig_box.show()
fig_violin.show()

             count        mean         std    min     25%    50%     75%  \
label_name                                                                 
ChatGPT       80.0   83.075000   85.982450   13.0   20.75   23.5  128.25   
Claude       240.0  419.875000  224.506882  161.0  191.00  424.5  610.25   
DeepSeek     320.0  179.418750   74.345923  105.0  131.00  153.0  198.25   
Gemini       160.0   55.643750   44.608619   15.0   24.00   33.0   89.25   
Grok          80.0  375.625000  115.086800  199.0  280.75  359.5  474.50   
Human       1520.0  450.869079  271.929139   74.0  186.00  452.5  584.00   

               max  
label_name          
ChatGPT      277.0  
Claude       800.0  
DeepSeek     473.0  
Gemini       197.0  
Grok         632.0  
Human       1910.0  


### LOAD DATASET AND SPLIT

In [11]:
# =============================================
#  LOAD DATASET + TRAIN/VAL SPLIT
# =============================================
df = pd.read_csv('/content/drive/MyDrive/malto/train.csv')

print('TOTAL SAMPLES:', len(df))

print('-'*50)
print('Label distribution:')
print(df['LABEL'].map(ID_TO_NAME).value_counts())

#split the dataset train + validation
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['LABEL']
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print('-'*50)
print(f'Train: {len(train_df)} | Val: {len(val_df)}')
print('Train distribution:')
print(train_df['LABEL'].map(ID_TO_NAME).value_counts())
print('-'*50)
print('Val distribution:')
print(val_df['LABEL'].map(ID_TO_NAME).value_counts())


TOTAL SAMPLES: 2400
--------------------------------------------------
Label distribution:
LABEL
Human       1520
DeepSeek     320
Claude       240
Gemini       160
ChatGPT       80
Grok          80
Name: count, dtype: int64
--------------------------------------------------
Train: 1920 | Val: 480
Train distribution:
LABEL
Human       1216
DeepSeek     256
Claude       192
Gemini       128
Grok          64
ChatGPT       64
Name: count, dtype: int64
--------------------------------------------------
Val distribution:
LABEL
Human       304
DeepSeek     64
Claude       48
Gemini       32
Grok         16
ChatGPT      16
Name: count, dtype: int64


### MODELS UTILS

In [13]:
# =============================================
#  DATASET CLASS
# =============================================
class AITextDataset(Dataset):

    # we perform the tokenization of the text to perform then the classification

    def __init__(self, df, tokenizer, max_len=512):
        self.texts     = df['TEXT'].values
        self.tokenizer = tokenizer
        self.max_len   = max_len

        # binary_label: 0=Human, 1=AI
        self.binary_labels = (df['LABEL'].values != HUMAN_ID).astype(int)

        # model_label: index [0..4], otherwise -1 per Human
        self.model_labels = np.array([
            AI_ID_TO_MODEL_IDX.get(int(lbl), -1)
            for lbl in df['LABEL'].values
        ])

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'binary_label':   torch.tensor(self.binary_labels[idx], dtype=torch.long),
            'model_label':    torch.tensor(self.model_labels[idx],  dtype=torch.long),
        }

In [15]:
# =============================================
#  MODEL ARCHITECTURE
# =============================================
# the architecture is defined as:
# a pretreinded transformer, followed by 2 different heads one for the HUMAN/AI classification
# the second one for  the model classification
#
class AIDetector(nn.Module):

    def __init__(self, model_name='roberta-base', num_models=5):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.3)

        self.binary_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(0.3), #to avoid overfitting
            nn.Linear(hidden // 2, 2) #final classification
        )
        self.model_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(hidden // 2, num_models)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(outputs.last_hidden_state[:, 0])
        return self.binary_head(cls), self.model_head(cls)




In [18]:
# =============================================
#  TRAINING + EVAL FUNCTIONS
# =============================================
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    bin_correct, bin_total = 0, 0
    mod_correct, mod_total = 0, 0

    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        binary_labels  = batch['binary_label'].to(device)
        model_labels   = batch['model_label'].to(device)

        optimizer.zero_grad()
        binary_logits, model_logits = model(input_ids, attention_mask)

        loss_binary = criterion_binary(binary_logits, binary_labels)

        # FIX: gradiente sul model_head solo per sample AI
        ai_mask = (binary_labels == 1)
        if ai_mask.sum() > 0:
            loss_model = criterion_model(model_logits[ai_mask], model_labels[ai_mask])
        else:
            loss_model = torch.tensor(0.0, device=device)

        loss = loss_binary + loss_model
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss  += loss.item()
        bin_preds    = binary_logits.argmax(dim=1)
        bin_correct += (bin_preds == binary_labels).sum().item()
        bin_total   += len(binary_labels)

        if ai_mask.sum() > 0:
            mod_preds    = model_logits[ai_mask].argmax(dim=1)
            mod_correct += (mod_preds == model_labels[ai_mask]).sum().item()
            mod_total   += ai_mask.sum().item()

    return (
        total_loss / len(loader),
        bin_correct / bin_total,
        mod_correct / mod_total if mod_total > 0 else 0.0
    )


def eval_epoch(model, loader):
    model.eval()
    total_loss = 0
    bin_correct, bin_total = 0, 0
    mod_correct, mod_total = 0, 0
    all_bin_preds, all_bin_true = [], []
    all_mod_preds, all_mod_true = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            binary_labels  = batch['binary_label'].to(device)
            model_labels   = batch['model_label'].to(device)

            binary_logits, model_logits = model(input_ids, attention_mask)

            loss_binary = criterion_binary(binary_logits, binary_labels)
            ai_mask = (binary_labels == 1)
            if ai_mask.sum() > 0:
                loss_model = criterion_model(model_logits[ai_mask], model_labels[ai_mask])
            else:
                loss_model = torch.tensor(0.0, device=device)

            total_loss  += (loss_binary + loss_model).item()
            bin_preds    = binary_logits.argmax(dim=1)
            bin_correct += (bin_preds == binary_labels).sum().item()
            bin_total   += len(binary_labels)
            all_bin_preds.extend(bin_preds.cpu().tolist())
            all_bin_true.extend(binary_labels.cpu().tolist())

            if ai_mask.sum() > 0:
                mod_preds    = model_logits[ai_mask].argmax(dim=1)
                mod_correct += (mod_preds == model_labels[ai_mask]).sum().item()
                mod_total   += ai_mask.sum().item()
                all_mod_preds.extend(mod_preds.cpu().tolist())
                all_mod_true.extend(model_labels[ai_mask].cpu().tolist())

    return (
        total_loss / len(loader),
        bin_correct / bin_total,
        mod_correct / mod_total if mod_total > 0 else 0.0,
        all_bin_preds, all_bin_true,
        all_mod_preds, all_mod_true
    )

In [20]:
# =============================================
#  INFERENCE SU TEST SET
# =============================================
class TestDataset(Dataset):

    def __init__(self, texts, tokenizer, max_len=512):
        self.texts     = texts.values
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
        }


def predict(model, texts, tokenizer, batch_size=16):
    dataset = TestDataset(texts, tokenizer)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model.eval()
    predictions = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            binary_logits, model_logits = model(input_ids, attention_mask)
            binary_preds = binary_logits.argmax(dim=1)
            model_preds  = model_logits.argmax(dim=1)

            for b, m in zip(binary_preds, model_preds):
                if b.item() == 0:
                    predictions.append('Human')
                else:
                    predictions.append(MODEL_IDX_TO_NAME[m.item()])

    return predictions




#SERCHING THE MODEL WITH THE BEST PARAMS

In [12]:
# =============================================
#  CLASS WEIGHTS
# =============================================

# Binary head weights (0=Human, 1=AI)
binary_labels_train = (train_df['LABEL'].values != HUMAN_ID).astype(int)

# weight = N/(classes*samples_in_classes)
binary_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=binary_labels_train
)

binary_class_weights = torch.tensor(binary_weights, dtype=torch.float).to(device)
print('Binary class weights [Human, AI]:', binary_class_weights)

#-------------------------------------------------------------------------------

# Model head weights (only to classify AI models, index [0..4])
ai_train_df = train_df[train_df['LABEL'] != HUMAN_ID].copy()
model_labels_train = ai_train_df['LABEL'].map(AI_ID_TO_MODEL_IDX).astype(int).values

# weight = N/(classes*samples_in_classes)
model_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_MODELS),
    y=model_labels_train
)

model_class_weights = torch.tensor(model_weights, dtype=torch.float).to(device)
print('Model class weights [ChatGPT, Gemini, Grok, Claude, DeepSeek]:', model_class_weights)

Binary class weights [Human, AI]: tensor([0.7895, 1.3636], device='cuda:0')
Model class weights [ChatGPT, Gemini, Grok, Claude, DeepSeek]: tensor([2.2000, 1.1000, 2.2000, 0.7333, 0.5500], device='cuda:0')


In [14]:
# =============================================
#  TOKENIZER + DATALOADERS
# =============================================
MODEL_NAME = 'answerdotai/ModernBERT-base' #'roberta-base'
MAX_LEN    = 512 # 256, 128
BATCH_SIZE = 16 # 8

# take the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# perform the transformation from text to token
train_dataset = AITextDataset(train_df, tokenizer, MAX_LEN)
val_dataset   = AITextDataset(val_df,   tokenizer, MAX_LEN)

# WeightedRandomSampler: to over-sample the rare classes(the same samples are used more then once during the training)
sample_weights = [
    binary_class_weights[int(label)].item()
    for label in train_dataset.binary_labels
]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Train batches: 120 | Val batches: 30


In [16]:
model = AIDetector(model_name=MODEL_NAME, num_models=NUM_MODELS).to(device)
print(f'Total params: {sum(p.numel() for p in model.parameters()):,}')

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total params: 149,607,559


In [17]:
# =============================================
#  LOSS, OPTIMIZER, SCHEDULER
# =============================================
NUM_EPOCHS = 7 # 5, 6
LR         = 1e-5 # 2e-5

#two loss function one for each head
criterion_binary = nn.CrossEntropyLoss(weight=binary_class_weights)
criterion_model  = nn.CrossEntropyLoss(weight=model_class_weights)

# L2 regularization
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps  = len(train_loader) * NUM_EPOCHS # the numer fo batches per numeber epochs to use whit the scheduler
warmup_steps = int(0.1 * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
print(f'Total steps: {total_steps} | Warmup steps: {warmup_steps}')

Total steps: 840 | Warmup steps: 84


In [19]:
# =============================================
#  TRAINING RUN
# =============================================
best_val_loss = float('inf')
history = []

print(f"{'Epoch':>5} | {'TrLoss':>8} | {'TrBinAcc':>9} | {'TrModAcc':>9} | {'VaLoss':>8} | {'VaBinAcc':>9} | {'VaModAcc':>9}")
print('-' * 75)

for epoch in range(1, NUM_EPOCHS + 1):

    tr_loss, tr_bin, tr_mod = train_epoch(model, train_loader, optimizer, scheduler)
    va_loss, va_bin, va_mod, bp, bt, mp, mt = eval_epoch(model, val_loader)

    history.append(dict(
        epoch=epoch,
        train_loss=tr_loss, val_loss=va_loss,
        train_bin_acc=tr_bin, val_bin_acc=va_bin,
        train_mod_acc=tr_mod, val_mod_acc=va_mod,
    ))

    print(f"{epoch:>5} | {tr_loss:>8.4f} | {tr_bin:>9.4f} | {tr_mod:>9.4f} | "
          f"{va_loss:>8.4f} | {va_bin:>9.4f} | {va_mod:>9.4f}")

    if va_loss < best_val_loss:
        best_val_loss = va_loss
        torch.save(model.state_dict(), 'best_model.pt')
        print(f"        -> best model saved (val_loss={va_loss:.4f})")

print('Training complete!')

Epoch |   TrLoss |  TrBinAcc |  TrModAcc |   VaLoss |  VaBinAcc |  VaModAcc
---------------------------------------------------------------------------


W0319 18:34:32.799000 3600 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode


    1 |   1.5567 |    0.8526 |    0.5367 |   0.5648 |    0.9938 |    0.8693
        -> best model saved (val_loss=0.5648)
    2 |   0.3919 |    0.9974 |    0.9134 |   0.2891 |    0.9958 |    0.9034
        -> best model saved (val_loss=0.2891)
    3 |   0.1523 |    1.0000 |    0.9701 |   0.1844 |    1.0000 |    0.9545
        -> best model saved (val_loss=0.1844)
    4 |   0.1212 |    1.0000 |    0.9819 |   0.2792 |    1.0000 |    0.9432
    5 |   0.0521 |    1.0000 |    0.9917 |   0.1342 |    1.0000 |    0.9602
        -> best model saved (val_loss=0.1342)
    6 |   0.0573 |    1.0000 |    0.9915 |   0.1529 |    1.0000 |    0.9602
    7 |   0.0323 |    1.0000 |    0.9959 |   0.1362 |    1.0000 |    0.9659
Training complete!


The best model was selected a the epoch 5 so for these reason whitout validation we will use 6 epoch to incress it

In [33]:
import pandas as pd
import plotly.express as px

# Convert history to DataFrame
df_hist = pd.DataFrame(history)

# Plot loss curves
fig = px.line(
    df_hist,
    x="epoch",
    y=["train_loss", "val_loss"],
    markers=True,
    title="Training vs Validation Loss",
    labels={
        "value": "Loss",
        "epoch": "Epoch",
        "variable": "Legend"
    }
)

best_epoch = df_hist.loc[df_hist["val_loss"].idxmin(), "epoch"]

fig.add_vline(
    x=best_epoch,
    line_dash="dash",
    line_color="green",
    annotation_text=f"Best Epoch: {best_epoch}"
)


fig.show()

In [32]:
# =============================================
#  EVALUATION
# =============================================
model.load_state_dict(torch.load('best_model.pt', map_location=device))
_, _, _, bin_preds, bin_true, mod_preds, mod_true = eval_epoch(model, val_loader)

print('=' * 55)
print('BINARY HEAD - Human vs AI')
print('=' * 55)
print(classification_report(bin_true, bin_preds, target_names=['Human', 'AI']))

print('=' * 55)
print('MODEL HEAD - Which AI (only AI samples)')
print('=' * 55)
model_names = [MODEL_IDX_TO_NAME[i] for i in range(NUM_MODELS)]
print(classification_report(mod_true, mod_preds, target_names=model_names))

print('Confusion matrix (model head):')
cm_df = pd.DataFrame(
    confusion_matrix(mod_true, mod_preds),
    index=model_names,
    columns=model_names
)
print(cm_df)

BINARY HEAD - Human vs AI
              precision    recall  f1-score   support

       Human       1.00      1.00      1.00       304
          AI       1.00      1.00      1.00       176

    accuracy                           1.00       480
   macro avg       1.00      1.00      1.00       480
weighted avg       1.00      1.00      1.00       480

MODEL HEAD - Which AI (only AI samples)
              precision    recall  f1-score   support

     ChatGPT       0.81      0.81      0.81        16
      Gemini       0.94      0.91      0.92        32
        Grok       1.00      1.00      1.00        16
      Claude       1.00      1.00      1.00        48
    DeepSeek       0.97      0.98      0.98        64

    accuracy                           0.96       176
   macro avg       0.94      0.94      0.94       176
weighted avg       0.96      0.96      0.96       176

Confusion matrix (model head):
          ChatGPT  Gemini  Grok  Claude  DeepSeek
ChatGPT        13       2     0      

In [34]:
# =============================================
# PERFORM THE PREDICTION ON THE TEST
# =============================================

test_df = pd.read_csv('/content/drive/MyDrive/malto/test.csv')
model.load_state_dict(torch.load('best_model.pt', map_location=device))
preds = predict(model, test_df['TEXT'], tokenizer)

test_df['LABEL'] = preds
test_df.to_csv('test_predictions.csv', index=False)
print('Predictions saved to test_predictions.csv')
print('Prediction distribution:')
print(pd.Series(preds).value_counts())


Predictions saved to test_predictions.csv
Prediction distribution:
Human       379
DeepSeek     80
Claude       60
Gemini       48
Grok         20
ChatGPT      13
Name: count, dtype: int64


In [36]:
# ==============================================
# SAVE THE PREDICTION ON THE CSV FORMAT WITH ID | LABELS
# ==============================================

df_test_res = pd.read_csv('test_predictions.csv')
label_map = {
    "Human": 0,
    "ChatGPT": 1,
    "Gemini": 2,
    "Grok": 3,
    "Claude": 4,
    "DeepSeek": 5
}

# Map the values
df_test_res["LABEL"] = df_test_res["LABEL"].map(label_map)

fix_format_1 = df_test_res.rename(columns={"Unnamed: 0":"ID"})

fix_format_2 = fix_format_1.drop(columns=["TEXT"])
fix_format_2.to_csv("testset_mapped1.csv", index=False)
fix_format_2.head(10)


,ID,LABEL
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0
5,5,5
6,6,0
7,7,0
8,8,5
9,9,3


# NO VALIDATION SET

In [5]:
# =============================================
#  LOAD DATASET + TRAIN
# =============================================
df = pd.read_csv('/content/drive/MyDrive/malto/train.csv')

print('TOTAL SAMPLES:', len(df))
print('Label distribution:')
print(df['LABEL'].map(ID_TO_NAME).value_counts())

#split the dataset train + validation
train_df = df
train_df = train_df.reset_index(drop=True)

print(f'Train: {len(train_df)} |')
print('Train distribution:')
print(train_df['LABEL'].map(ID_TO_NAME).value_counts())

TOTAL SAMPLES: 2400
Label distribution:
LABEL
Human       1520
DeepSeek     320
Claude       240
Gemini       160
ChatGPT       80
Grok          80
Name: count, dtype: int64
Train: 2400 |
Train distribution:
LABEL
Human       1520
DeepSeek     320
Claude       240
Gemini       160
ChatGPT       80
Grok          80
Name: count, dtype: int64


In [6]:
# =============================================
#  CLASS WEIGHTS
# =============================================

# Binary head weights (0=Human, 1=AI)
binary_labels_train = (train_df['LABEL'].values != HUMAN_ID).astype(int)

# weight = N/(classes*samples_in_classes)
binary_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=binary_labels_train
)

binary_class_weights = torch.tensor(binary_weights, dtype=torch.float).to(device)
print('Binary class weights [Human, AI]:', binary_class_weights)

#-------------------------------------------------------------------------------

# Model head weights (only to classify AI models, index [0..4])
ai_train_df = train_df[train_df['LABEL'] != HUMAN_ID].copy()
model_labels_train = ai_train_df['LABEL'].map(AI_ID_TO_MODEL_IDX).astype(int).values

# weight = N/(classes*samples_in_classes)
model_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_MODELS),
    y=model_labels_train
)

model_class_weights = torch.tensor(model_weights, dtype=torch.float).to(device)
print('Model class weights [ChatGPT, Gemini, Grok, Claude, DeepSeek]:', model_class_weights)

Binary class weights [Human, AI]: tensor([0.7895, 1.3636], device='cuda:0')
Model class weights [ChatGPT, Gemini, Grok, Claude, DeepSeek]: tensor([2.2000, 1.1000, 2.2000, 0.7333, 0.5500], device='cuda:0')


In [8]:
# =============================================
#  TOKENIZER + DATALOADERS
# =============================================
MODEL_NAME = 'answerdotai/ModernBERT-base'
MAX_LEN    = 512
BATCH_SIZE = 16

# take the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# perform the transformation from text to token
train_dataset = AITextDataset(train_df, tokenizer, MAX_LEN)

# WeightedRandomSampler: to over-sample the rare classes(the same samples are used more then once during the training)
sample_weights = [
    binary_class_weights[int(label)].item()
    for label in train_dataset.binary_labels
]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)

print(f'Train batches: {len(train_loader)} | ')

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Train batches: 150 | 


In [9]:
# =============================================
#  MODEL ARCHITECTURE
# =============================================
#

model = AIDetector(model_name=MODEL_NAME, num_models=NUM_MODELS).to(device)
print(f'Parametri totali: {sum(p.numel() for p in model.parameters()):,}')

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Parametri totali: 149,607,559


In [10]:
# =============================================
#  LOSS, OPTIMIZER, SCHEDULER
# =============================================
NUM_EPOCHS = 6
LR         = 1e-5

#two loss function one for each head
criterion_binary = nn.CrossEntropyLoss(weight=binary_class_weights)
criterion_model  = nn.CrossEntropyLoss(weight=model_class_weights)

# L2 regularization
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps  = len(train_loader) * NUM_EPOCHS # the numer fo batches per numeber epochs to use whit the scheduler
warmup_steps = int(0.1 * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
print(f'Total steps: {total_steps} | Warmup steps: {warmup_steps}')

Total steps: 900 | Warmup steps: 90


In [13]:
# =============================================
#  TRAINING RUN
# =============================================
best_val_loss = float('inf')
history = []

print(f"{'Epoch':>5} | {'TrLoss':>8} | {'TrBinAcc':>9} | {'TrModAcc':>9} | {'VaLoss':>8} | {'VaBinAcc':>9} | {'VaModAcc':>9}")
print('-' * 75)

for epoch in range(1, NUM_EPOCHS + 1):

    tr_loss, tr_bin, tr_mod = train_epoch(model, train_loader, optimizer, scheduler)

    history.append(dict(
        epoch=epoch,
        train_loss=tr_loss,
        train_bin_acc=tr_bin,
        train_mod_acc=tr_mod,
    ))

    print(f"{epoch:>5} | {tr_loss:>8.4f} | {tr_bin:>9.4f} | {tr_mod:>9.4f} | " )

    torch.save(model.state_dict(), f'best_model_epoch_{epoch}.pt')

print('Training complete!')

Epoch |   TrLoss |  TrBinAcc |  TrModAcc |   VaLoss |  VaBinAcc |  VaModAcc
---------------------------------------------------------------------------
    1 |   0.3602 |    0.9988 |    0.9178 | 
    2 |   0.1432 |    0.9992 |    0.9726 | 
    3 |   0.1389 |    0.9996 |    0.9803 | 
    4 |   0.0615 |    0.9996 |    0.9926 | 
    5 |   0.0203 |    1.0000 |    0.9983 | 
    6 |   0.0160 |    0.9992 |    0.9983 | 
Training complete!


In [ ]:
# =============================================
#  EVALUATION DETTAGLIATA
# =============================================
model.load_state_dict(torch.load('best_model.pt', map_location=device))
_, _, _, bin_preds, bin_true, mod_preds, mod_true = eval_epoch(model, train_loss)

print('=' * 55)
print('BINARY HEAD - Human vs AI')
print('=' * 55)
print(classification_report(bin_true, bin_preds, target_names=['Human', 'AI']))

print('=' * 55)
print('MODEL HEAD - Which AI (only AI samples)')
print('=' * 55)
model_names = [MODEL_IDX_TO_NAME[i] for i in range(NUM_MODELS)]
print(classification_report(mod_true, mod_preds, target_names=model_names))

print('Confusion matrix (model head):')
cm_df = pd.DataFrame(
    confusion_matrix(mod_true, mod_preds),
    index=model_names,
    columns=model_names
)
print(cm_df)

BINARY HEAD - Human vs AI
              precision    recall  f1-score   support

       Human       1.00      1.00      1.00       304
          AI       1.00      1.00      1.00       176

    accuracy                           1.00       480
   macro avg       1.00      1.00      1.00       480
weighted avg       1.00      1.00      1.00       480

MODEL HEAD - Which AI (only AI samples)
              precision    recall  f1-score   support

     ChatGPT       0.82      0.88      0.85        16
      Gemini       0.97      0.91      0.94        32
        Grok       1.00      1.00      1.00        16
      Claude       0.98      1.00      0.99        48
    DeepSeek       1.00      1.00      1.00        64

    accuracy                           0.97       176
   macro avg       0.95      0.96      0.95       176
weighted avg       0.97      0.97      0.97       176

Confusion matrix (model head):
          ChatGPT  Gemini  Grok  Claude  DeepSeek
ChatGPT        14       1     0      

In [15]:
# =============================================
#  INFERENCE SU TEST SET
# =============================================

test_df = pd.read_csv('/content/drive/MyDrive/malto/test.csv')
model.load_state_dict(torch.load('best_model_epoch_6.pt', map_location=device))
preds = predict(model, test_df['TEXT'], tokenizer)

test_df['LABEL'] = preds
test_df.to_csv('test_predictions.csv', index=False)
print('Predictions saved to test_predictions.csv')
print('Prediction distribution:')
print(pd.Series(preds).value_counts())

Predictions saved to test_predictions.csv
Prediction distribution:
Human       380
DeepSeek     80
Claude       61
Gemini       37
ChatGPT      22
Grok         20
Name: count, dtype: int64


In [21]:
df_test_res = pd.read_csv('test_predictions.csv')
label_map = {
    "Human": 0,
    "ChatGPT": 1,
    "Gemini": 2,
    "Grok": 3,
    "Claude": 4,
    "DeepSeek": 5
}

# Map the values
df_test_res["LABEL"] = df_test_res["LABEL"].map(label_map)
fix_format_1 = df_test_res.rename(columns={"Unnamed: 0":"ID"})
fix_format_2 = fix_format_1.drop(columns=["TEXT"])
# Save the result
fix_format_2.to_csv("testset_mapped.csv", index=False)


In [22]:
fix_format_2.head(10 )

,ID,LABEL
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0
5,5,5
6,6,0
7,7,0
8,8,5
9,9,3
